In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

PROJECT_PATH = "/content/drive/MyDrive/Face_Emotion_Project"

print("프로젝트 폴더 확인:", os.path.exists(PROJECT_PATH))
print("폴더 내부:", os.listdir(PROJECT_PATH))

In [ ]:
import os

original_path = os.path.join(PROJECT_PATH, "original")

for dataset in ["WSEFEP", "V_Expression"]:
    dataset_path = os.path.join(original_path, dataset)

    files = os.listdir(dataset_path)

    images = [
        f for f in files
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    print(f"\n[{dataset}]")
    print("전체 파일:", len(files))
    print("이미지 파일:", len(images))
    print("파일명 예시:")

    for name in images[:5]:
        print(" -", name)

In [ ]:
import os
from collections import Counter

# 1. V_Expression 감정별 이미지 수 확인
v_path = os.path.join(PROJECT_PATH, "original", "V_Expression")

emotion_codes = {
    "E01": "Neutral",
    "E02": "Happy",
    "E03": "Sad",
    "E04": "Fear",
    "E05": "Angry",
    "E06": "Surprise",
    "E07": "Disgust"
}

v_counts = Counter()

for filename in os.listdir(v_path):
    if filename.lower().endswith((".jpg", ".jpeg", ".png")):
        for code, emotion in emotion_codes.items():
            if f"_{code}_" in filename:
                v_counts[emotion] += 1
                break

print("===== V_Expression 감정별 이미지 수 =====")

for emotion in emotion_codes.values():
    print(f"{emotion}: {v_counts[emotion]}장")

# 2. WSEFEP 엑셀 파일 확인
w_path = os.path.join(PROJECT_PATH, "original", "WSEFEP")

print("\n===== WSEFEP 엑셀 파일 =====")

for filename in os.listdir(w_path):
    if filename.lower().endswith((".xlsx", ".xls", ".csv")):
        print(filename)

In [ ]:
import pandas as pd
import os

excel_path = os.path.join(
    PROJECT_PATH,
    "original",
    "WSEFEP",
    "WSEFEP - norms & FACS.xlsx"
)

# 엑셀 시트 목록 확인
excel_data = pd.ExcelFile(excel_path)

print("===== 엑셀 시트 목록 =====")
print(excel_data.sheet_names)

# 각 시트의 열 이름과 데이터 확인
for sheet in excel_data.sheet_names:
    df = pd.read_excel(excel_path, sheet_name=sheet)

    print(f"\n===== 시트: {sheet} =====")
    print("열 이름:", df.columns.tolist())
    print(df.head(5).to_string(index=False))

In [ ]:
import pandas as pd
import os

excel_path = os.path.join(
    PROJECT_PATH,
    "original",
    "WSEFEP",
    "WSEFEP - norms & FACS.xlsx"
)

df = pd.read_excel(excel_path)

# 감정 이름 통일
emotion_mapping = {
    "neutral": "Neutral",
    "joy": "Happy",
    "sadness": "Sad",
    "fear": "Fear",
    "anger": "Angry",
    "surprise": "Surprise",
    "disgust": "Disgust"
}

df["Emotion"] = df["Display"].astype(str).str.strip().str.lower().map(emotion_mapping)

print("===== WSEFEP 감정별 이미지 수 =====")
print(df["Emotion"].value_counts().reindex(emotion_mapping.values(), fill_value=0))

print("\n전체 엑셀 이미지 수:", len(df))
print("감정 라벨이 확인된 이미지 수:", df["Emotion"].notna().sum())

In [ ]:
import os
import shutil
import pandas as pd

# 경로 설정
original_path = os.path.join(PROJECT_PATH, "original")
output_path = os.path.join(PROJECT_PATH, "processed")

w_path = os.path.join(original_path, "WSEFEP")
v_path = os.path.join(original_path, "V_Expression")

# 감정 이름 통일
emotion_mapping = {
    "neutral": "Neutral",
    "joy": "Happy",
    "sadness": "Sad",
    "fear": "Fear",
    "anger": "Angry",
    "surprise": "Surprise",
    "disgust": "Disgust"
}

v_mapping = {
    "E01": "Neutral",
    "E02": "Happy",
    "E03": "Sad",
    "E04": "Fear",
    "E05": "Angry",
    "E06": "Surprise",
    "E07": "Disgust"
}

# 감정 폴더 생성
for emotion in emotion_mapping.values():
    os.makedirs(os.path.join(output_path, emotion), exist_ok=True)

# WSEFEP 라벨 읽기
excel_path = os.path.join(
    w_path, "WSEFEP - norms & FACS.xlsx"
)
df = pd.read_excel(excel_path)

copied = {emotion: 0 for emotion in emotion_mapping.values()}
missing = []

# WSEFEP 이미지 복사
for _, row in df.iterrows():
    filename = str(row["Picture ID"]).strip()
    emotion = emotion_mapping.get(
        str(row["Display"]).strip().lower()
    )

    source = os.path.join(w_path, filename)

    if emotion and os.path.isfile(source):
        destination = os.path.join(
            output_path,
            emotion,
            "WSEFEP_" + filename
        )
        shutil.copy2(source, destination)
        copied[emotion] += 1
    else:
        missing.append(filename)

# V_Expression 이미지 복사
for filename in os.listdir(v_path):
    if not filename.lower().endswith((".jpg", ".jpeg", ".png")):
        continue

    emotion = None

    for code, label in v_mapping.items():
        if f"_{code}_" in filename:
            emotion = label
            break

    if emotion:
        source = os.path.join(v_path, filename)
        destination = os.path.join(
            output_path,
            emotion,
            "VEXP_" + filename
        )
        shutil.copy2(source, destination)
        copied[emotion] += 1

# 결과 출력
print("===== 통합 완료 =====")

for emotion, count in copied.items():
    print(f"{emotion}: {count}장")

print("전체 복사 수:", sum(copied.values()))
print("WSEFEP 누락 이미지:", len(missing))

In [ ]:
import os
import pandas as pd

w_path = os.path.join(PROJECT_PATH, "original", "WSEFEP")
excel_path = os.path.join(w_path, "WSEFEP - norms & FACS.xlsx")

df = pd.read_excel(excel_path)

# 실제 폴더의 파일 목록
actual_files = [
    f for f in os.listdir(w_path)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
]

# 대소문자를 무시하고 비교
actual_lookup = {f.lower(): f for f in actual_files}

missing_files = []

for _, row in df.iterrows():
    picture_id = str(row["Picture ID"]).strip()

    if picture_id.lower() not in actual_lookup:
        missing_files.append({
            "Picture ID": picture_id,
            "감정": row["Display"]
        })

print("===== 누락된 이미지 =====")
for item in missing_files:
    print(item)

print("\n누락 개수:", len(missing_files))

# 엑셀에 등록되지 않은 실제 이미지도 확인
excel_ids = set(
    df["Picture ID"].astype(str).str.strip().str.lower()
)
unlisted_files = [
    f for f in actual_files if f.lower() not in excel_ids
]

print("\n===== 엑셀에 등록되지 않은 이미지 =====")
for f in unlisted_files:
    print(f)

In [ ]:
from PIL import Image
import os

w_path = os.path.join(PROJECT_PATH, "original", "WSEFEP")

check_files = [
    "MG_0850.jpg",
    "AD_8268.jpg",
    "SO_0053.jpg",
    "JS_0449.jpg"
]

for filename in check_files:
    path = os.path.join(w_path, filename)

    if os.path.exists(path):
        img = Image.open(path)

        print(filename)
        print("크기:", img.size)
        print("색상 모드:", img.mode)
        display(img.resize((180, 180)))

In [ ]:
import os
import shutil
import pandas as pd
from collections import Counter

original_path = os.path.join(PROJECT_PATH, "original")
output_path = os.path.join(PROJECT_PATH, "processed")

w_path = os.path.join(original_path, "WSEFEP")
v_path = os.path.join(original_path, "V_Expression")

# 기존 통합 폴더 초기화
if os.path.exists(output_path):
    shutil.rmtree(output_path)

emotions = [
    "Neutral", "Happy", "Sad", "Fear",
    "Angry", "Surprise", "Disgust"
]

for emotion in emotions:
    os.makedirs(os.path.join(output_path, emotion))

# WSEFEP 감정 매핑
w_mapping = {
    "neutral": "Neutral",
    "joy": "Happy",
    "sadness": "Sad",
    "fear": "Fear",
    "anger": "Angry",
    "surprise": "Surprise",
    "disgust": "Disgust"
}

# V_Expression 감정 매핑
v_mapping = {
    "E01": "Neutral",
    "E02": "Happy",
    "E03": "Sad",
    "E04": "Fear",
    "E05": "Angry",
    "E06": "Surprise",
    "E07": "Disgust"
}

counts = Counter()
missing = []

# WSEFEP 복사
excel_path = os.path.join(
    w_path, "WSEFEP - norms & FACS.xlsx"
)
df = pd.read_excel(excel_path)

actual_files = {
    f.lower(): f for f in os.listdir(w_path)
}

for _, row in df.iterrows():
    filename = str(row["Picture ID"]).strip()
    emotion = w_mapping.get(str(row["Display"]).strip().lower())

    actual_name = actual_files.get(filename.lower())

    if emotion and actual_name:
        src = os.path.join(w_path, actual_name)
        dst = os.path.join(
            output_path, emotion, "WSEFEP_" + actual_name
        )
        shutil.copy2(src, dst)
        counts[emotion] += 1
    else:
        missing.append(filename)

# V_Expression 복사
for filename in os.listdir(v_path):
    if not filename.lower().endswith((".jpg", ".jpeg", ".png")):
        continue

    emotion = next(
        (label for code, label in v_mapping.items()
         if f"_{code}_" in filename),
        None
    )

    if emotion:
        src = os.path.join(v_path, filename)
        dst = os.path.join(
            output_path, emotion, "VEXP_" + filename
        )
        shutil.copy2(src, dst)
        counts[emotion] += 1

print("===== 최종 통합 결과 =====")

for emotion in emotions:
    print(f"{emotion}: {counts[emotion]}장")

print("총 이미지:", sum(counts.values()))
print("WSEFEP 누락:", len(missing))
print("누락 파일:", missing)

In [ ]:
from PIL import Image
import os
from collections import Counter

image_extensions = (".jpg", ".jpeg", ".png")

total_images = 0
rgb_images = 0
damaged_images = []
image_sizes = Counter()

for emotion in emotions:
    folder = os.path.join(output_path, emotion)

    for filename in os.listdir(folder):
        if not filename.lower().endswith(image_extensions):
            continue

        path = os.path.join(folder, filename)
        total_images += 1

        try:
            with Image.open(path) as img:
                image_sizes[img.size] += 1

                if img.mode == "RGB":
                    rgb_images += 1

        except Exception:
            damaged_images.append(filename)

print("===== 이미지 검사 결과 =====")
print("전체 이미지:", total_images)
print("RGB 이미지:", rgb_images)
print("손상된 이미지:", len(damaged_images))

print("\n===== 이미지 크기 분포 =====")
for size, count in image_sizes.most_common(10):
    print(f"{size}: {count}장")

print("\n손상된 파일:", damaged_images)

In [ ]:
import os
import re
from collections import Counter

for emotion in emotions:
    folder = os.path.join(output_path, emotion)

    subjects = set()

    for filename in os.listdir(folder):
        if filename.startswith("VEXP_"):
            match = re.match(r"VEXP_(V\d+)_", filename)

            if match:
                subjects.add(match.group(1))

    print(f"{emotion}: V_Expression 인물 {len(subjects)}명")

In [ ]:
import pandas as pd
import os

excel_path = os.path.join(
    PROJECT_PATH,
    "original",
    "WSEFEP",
    "WSEFEP - norms & FACS.xlsx"
)

df = pd.read_excel(excel_path)

print("===== WSEFEP 인물별 이미지 수 =====")
print(df["Displayer ID"].value_counts())

print("\n고유 인물 수:", df["Displayer ID"].nunique())

In [ ]:
import os
import re
import pandas as pd
from sklearn.model_selection import train_test_split

# 데이터 경로
output_path = os.path.join(PROJECT_PATH, "processed")
split_path = os.path.join(PROJECT_PATH, "split")
os.makedirs(split_path, exist_ok=True)

emotions = [
    "Angry", "Disgust", "Fear",
    "Happy", "Neutral", "Sad", "Surprise"
]

records = []

# 이미지 파일 목록 수집
for emotion in emotions:
    folder = os.path.join(output_path, emotion)

    for filename in os.listdir(folder):
        if not filename.lower().endswith((".jpg", ".jpeg", ".png")):
            continue

        if filename.startswith("VEXP_"):
            dataset = "V_Expression"
            match = re.match(r"VEXP_(V\d+)_", filename)
            subject = match.group(1) if match else None

        elif filename.startswith("WSEFEP_"):
            dataset = "WSEFEP"
            # WSEFEP 파일명 예: WSEFEP_AD_7950.jpg
            match = re.match(r"WSEFEP_([A-Z]+\d*)_", filename)
            subject = match.group(1) if match else None

        else:
            continue

        records.append({
            "path": os.path.join(folder, filename),
            "emotion": emotion,
            "dataset": dataset,
            "subject": subject
        })

data = pd.DataFrame(records)

print("전체 이미지:", len(data))
print("인물 ID 누락:", data["subject"].isna().sum())
print(data.groupby("dataset")["subject"].nunique())

In [ ]:
# 데이터셋별로 인물 단위 분리
train_parts = []
val_parts = []
test_parts = []

for dataset_name, group in data.groupby("dataset"):

    subjects = group["subject"].unique()

    train_subjects, temp_subjects = train_test_split(
        subjects,
        test_size=0.30,
        random_state=42
    )

    val_subjects, test_subjects = train_test_split(
        temp_subjects,
        test_size=0.50,
        random_state=42
    )

    train_parts.append(
        group[group["subject"].isin(train_subjects)]
    )

    val_parts.append(
        group[group["subject"].isin(val_subjects)]
    )

    test_parts.append(
        group[group["subject"].isin(test_subjects)]
    )

train_df = pd.concat(train_parts).reset_index(drop=True)
val_df = pd.concat(val_parts).reset_index(drop=True)
test_df = pd.concat(test_parts).reset_index(drop=True)

# CSV 저장
train_df.to_csv(os.path.join(split_path, "train.csv"), index=False)
val_df.to_csv(os.path.join(split_path, "validation.csv"), index=False)
test_df.to_csv(os.path.join(split_path, "test.csv"), index=False)

print("===== 데이터 분리 결과 =====")
print("학습:", len(train_df))
print("검증:", len(val_df))
print("테스트:", len(test_df))
print("전체:", len(train_df) + len(val_df) + len(test_df))

In [ ]:
for name, df_part in [
    ("학습", train_df),
    ("검증", val_df),
    ("테스트", test_df)
]:
    print(f"\n===== {name} 데이터 =====")
    print(df_part.groupby(
        ["dataset", "emotion"]
    ).size().unstack(fill_value=0))

# 인물 중복 여부 확인
train_ids = set(zip(train_df["dataset"], train_df["subject"]))
val_ids = set(zip(val_df["dataset"], val_df["subject"]))
test_ids = set(zip(test_df["dataset"], test_df["subject"]))

print("\n===== 인물 중복 검사 =====")
print("학습-검증:", len(train_ids & val_ids))
print("학습-테스트:", len(train_ids & test_ids))
print("검증-테스트:", len(val_ids & test_ids))

In [ ]:
import torch

print("PyTorch 버전:", torch.__version__)
print("GPU 사용 가능:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU 이름:", torch.cuda.get_device_name(0))
else:
    print("GPU가 없습니다. Colab 런타임 유형에서 T4 GPU를 선택해 주세요.")

In [ ]:
import torch

print("PyTorch 버전:", torch.__version__)
print("GPU 사용 가능:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU 이름:", torch.cuda.get_device_name(0))

In [ ]:
import torch
import torchvision
import sklearn
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("라이브러리 준비 완료!")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT_PATH = "/content/drive/MyDrive/Face_Emotion_Project"

split_path = os.path.join(PROJECT_PATH, "split")
output_path = os.path.join(PROJECT_PATH, "processed")

print("프로젝트 폴더:", os.listdir(PROJECT_PATH))
print("split 폴더 존재:", os.path.exists(split_path))
print("processed 폴더 존재:", os.path.exists(output_path))

In [ ]:
import torch
import pandas as pd
import os

from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# 감정 클래스 순서 고정
class_names = [
    "Angry",
    "Disgust",
    "Fear",
    "Happy",
    "Neutral",
    "Sad",
    "Surprise"
]

class EmotionDataset(Dataset):
    def __init__(self, csv_path, transform=None):
        self.data = pd.read_csv(csv_path)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]

        image_path = row["path"]
        emotion = row["emotion"]

        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = class_names.index(emotion)

        return image, label

# 학습 데이터 증강
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# 검증 및 테스트 데이터
eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# 데이터셋 생성
train_dataset = EmotionDataset(
    os.path.join(split_path, "train.csv"),
    train_transform
)

val_dataset = EmotionDataset(
    os.path.join(split_path, "validation.csv"),
    eval_transform
)

test_dataset = EmotionDataset(
    os.path.join(split_path, "test.csv"),
    eval_transform
)

# GPU 학습용 데이터 로더
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("===== 데이터 로더 생성 완료 =====")
print("학습:", len(train_dataset))
print("검증:", len(val_dataset))
print("테스트:", len(test_dataset))

In [ ]:
images, labels = next(iter(train_loader))

print("이미지 배치 크기:", images.shape)
print("라벨 배치 크기:", labels.shape)
print("첫 번째 이미지 감정:", class_names[labels[0].item()])
print("데이터 타입:", images.dtype)

In [ ]:
import torch
import torch.nn as nn
from torchvision import models

# GPU 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# EfficientNet-B0 사전학습 모델 불러오기
weights = models.EfficientNet_B0_Weights.DEFAULT
model = models.efficientnet_b0(weights=weights)

# 마지막 분류층을 7개 감정으로 변경
num_features = model.classifier[1].in_features

model.classifier[1] = nn.Linear(
    num_features,
    7
)

# GPU로 모델 이동
model = model.to(device)

print("===== EfficientNet-B0 생성 완료 =====")
print("사용 장치:", device)
print("출력 클래스:", len(class_names))
print("감정 목록:", class_names)

In [ ]:
print(model.classifier)

In [ ]:
import torch.nn as nn
import torch.optim as optim

# 손실함수
criterion = nn.CrossEntropyLoss()

# AdamW 옵티마이저
optimizer = optim.AdamW(
    model.parameters(),
    lr=0.0001,
    weight_decay=0.0001
)

# 학습 횟수
num_epochs = 20

print("손실함수:", criterion)
print("옵티마이저:", optimizer)
print("학습 횟수:", num_epochs)

In [ ]:
import copy
import torch
import os

best_val_acc = 0.0

history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": []
}

model_path = os.path.join(PROJECT_PATH, "model")
os.makedirs(model_path, exist_ok=True)

for epoch in range(num_epochs):

    # ===== 학습 =====
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_loss = running_loss / total
    train_acc = correct / total

    # ===== 검증 =====
    model.eval()

    val_loss_sum = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss_sum += loss.item() * images.size(0)

            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_loss = val_loss_sum / val_total
    val_acc = val_correct / val_total

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Train Loss: {train_loss:.4f} "
        f"Train Acc: {train_acc:.4f} "
        f"Val Loss: {val_loss:.4f} "
        f"Val Acc: {val_acc:.4f}"
    )

    # 최고 검증 정확도 모델 저장
    if val_acc > best_val_acc:
        best_val_acc = val_acc

        torch.save({
            "model_state_dict": copy.deepcopy(model.state_dict()),
            "class_names": class_names,
            "epoch": epoch + 1,
            "val_acc": val_acc
        }, os.path.join(model_path, "best_emotion_model.pth"))

        print("  -> 최고 모델 저장 완료!")

print("\n최고 검증 정확도:", best_val_acc)

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# 저장된 최고 모델 불러오기
checkpoint = torch.load(
    os.path.join(PROJECT_PATH, "model", "best_emotion_model.pth"),
    map_location=device,
    weights_only=False
)

model.load_state_dict(checkpoint["model_state_dict"])
class_names = checkpoint["class_names"]

print("불러온 모델 에폭:", checkpoint["epoch"])
print("최고 검증 정확도:", checkpoint["val_acc"])

# 테스트 데이터 평가
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)

        outputs = model(images)
        predictions = torch.argmax(outputs, dim=1)

        all_preds.extend(predictions.cpu().numpy())
        all_labels.extend(labels.numpy())

# 정확도
test_acc = accuracy_score(all_labels, all_preds)

print("\n===== 테스트 결과 =====")
print(f"테스트 정확도: {test_acc * 100:.2f}%")

# 정밀도, 재현율, F1-score
print("\n===== 감정별 성능 =====")
print(classification_report(
    all_labels,
    all_preds,
    labels=list(range(len(class_names))),
    target_names=class_names,
    zero_division=0,
    digits=4
))

# 혼동행렬
cm = confusion_matrix(
    all_labels,
    all_preds,
    labels=list(range(len(class_names)))
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names
)

fig, ax = plt.subplots(figsize=(9, 8))
disp.plot(cmap="Blues", ax=ax, values_format="d")
plt.title("Emotion Classification - Test Confusion Matrix")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
!pip install ultralytics

In [ ]:
from ultralytics import YOLO

# YOLO 얼굴 전용 모델 다운로드 및 로드
face_detector = YOLO(
    "https://github.com/akanametov/yolo-face/releases/download/1.0.0/yolov11n-face.pt"
)

print("YOLO 얼굴 감지 모델 로드 완료!")

In [ ]:
from google.colab import files
import cv2
import matplotlib.pyplot as plt

# 1. 사진 업로드
uploaded = files.upload()
image_path = list(uploaded.keys())[-1]

print("현재 분석 사진:", image_path)

# 2. 이미지 불러오기
image = cv2.imread(image_path)

if image is None:
    print("이미지를 불러오지 못했습니다.")

else:
    h, w = image.shape[:2]

    # 3. YOLO 얼굴 감지
    results = face_detector(image)

    # 4. 원본 이미지 복사
    result_image = image.copy()

    # 5. 감지된 얼굴 박스를 처음부터 15% 확장
    for box in results[0].boxes.xyxy.cpu().numpy():

        x1, y1, x2, y2 = map(int, box)

        face_w = x2 - x1
        face_h = y2 - y1

        # 좌우 및 상하 각각 15% 확장
        margin_x = int(face_w * 0.15)
        margin_y = int(face_h * 0.15)

        x1 = max(0, x1 - margin_x)
        y1 = max(0, y1 - margin_y)
        x2 = min(w, x2 + margin_x)
        y2 = min(h, y2 + margin_y)

        # 확장된 얼굴 박스 표시
        cv2.rectangle(
            result_image,
            (x1, y1),
            (x2, y2),
            (255, 0, 0),
            2
        )

        # 감지 신뢰도 표시
        confidence = float(results[0].boxes.conf[0])
        cv2.putText(
            result_image,
            f"Face {confidence:.2f}",
            (x1, max(y1 - 10, 20)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (255, 0, 0),
            2
        )

    # 6. 결과 출력
    result_rgb = cv2.cvtColor(
        result_image, cv2.COLOR_BGR2RGB
    )

    plt.figure(figsize=(9, 7))
    plt.imshow(result_rgb)
    plt.axis("off")
    plt.title("YOLO Face Detection")
    plt.show()

    print("감지된 얼굴 수:", len(results[0].boxes))

In [ ]:
import torch
from torchvision import transforms
from PIL import Image
import numpy as np

# EfficientNet 입력 전처리
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# 잘라낸 얼굴을 RGB 이미지로 변환
face_pil = Image.fromarray(face_rgb)

# 텐서 변환
face_tensor = transform(face_pil).unsqueeze(0).to(device)

# 감정 예측
model.eval()

with torch.no_grad():
    output = model(face_tensor)
    probabilities = torch.softmax(output, dim=1)[0]

# 가장 높은 확률의 감정
predicted_idx = torch.argmax(probabilities).item()

print("===== 감정 분석 결과 =====")
print(f"예측 감정: {class_names[predicted_idx]}")
print(f"예측 확률: {probabilities[predicted_idx].item() * 100:.2f}%")

# 전체 감정별 확률
for name, prob in zip(class_names, probabilities):
    print(f"{name}: {prob.item() * 100:.2f}%")

In [ ]:
import cv2
import torch
import matplotlib.pyplot as plt

from google.colab import files
from PIL import Image
from torchvision import transforms

# 1. 새 사진 업로드
uploaded = files.upload()
image_path = list(uploaded.keys())[-1]

image = cv2.imread(image_path)
h, w = image.shape[:2]

# 2. YOLO 얼굴 감지
results = face_detector(image)

if len(results[0].boxes) == 0:
    print("얼굴을 감지하지 못했습니다.")

else:
    # 가장 큰 얼굴 선택
    boxes = results[0].boxes.xyxy.cpu().numpy()
    areas = [
        (b[2] - b[0]) * (b[3] - b[1])
        for b in boxes
    ]

    box = boxes[areas.index(max(areas))]
    x1, y1, x2, y2 = map(int, box)

    # 3. 얼굴 영역 15% 확장
    face_w = x2 - x1
    face_h = y2 - y1

    margin_x = int(face_w * 0.15)
    margin_y = int(face_h * 0.15)

    x1 = max(0, x1 - margin_x)
    y1 = max(0, y1 - margin_y)
    x2 = min(w, x2 + margin_x)
    y2 = min(h, y2 + margin_y)

    # 4. 확장된 영역으로 얼굴 자르기
    face_crop = image[y1:y2, x1:x2]

    face_rgb = cv2.cvtColor(
        face_crop, cv2.COLOR_BGR2RGB
    )

    # 5. 확장된 박스 표시
    result_image = image.copy()

    cv2.rectangle(
        result_image,
        (x1, y1),
        (x2, y2),
        (255, 0, 0),
        2
    )

    plt.figure(figsize=(8, 5))
    plt.imshow(cv2.cvtColor(result_image, cv2.COLOR_BGR2RGB))
    plt.title("YOLO Face Detection (+15%)")
    plt.axis("off")
    plt.show()

    # 6. 실제 EfficientNet 입력 이미지 확인
    plt.figure(figsize=(4, 4))
    plt.imshow(face_rgb)
    plt.title("Actual Face Input")
    plt.axis("off")
    plt.show()

    # 7. EfficientNet 전처리
    face_pil = Image.fromarray(face_rgb)

    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])

    face_tensor = transform(face_pil).unsqueeze(0).to(device)

    # 8. 감정 예측
    model.eval()

    with torch.no_grad():
        output = model(face_tensor)
        probabilities = torch.softmax(output, dim=1)[0]

    predicted_idx = probabilities.argmax().item()

    print("\n===== 감정 분석 결과 =====")
    print("예측 감정:", class_names[predicted_idx])
    print(f"예측 점수: {probabilities[predicted_idx].item()*100:.2f}%")

    for name, prob in zip(class_names, probabilities):
        print(f"{name}: {prob.item()*100:.2f}%")

In [ ]:
!pip install mediapipe

In [ ]:
import mediapipe as mp

print("MediaPipe 버전:", mp.__version__)
print("MediaPipe 설치 완료!")

In [ ]:
import mediapipe as mp

print("MediaPipe 버전:", mp.__version__)
print("설치 경로:", mp.__file__)
print("solutions 사용 가능:", hasattr(mp, "solutions"))

In [ ]:
import urllib.request

model_url = "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task"

model_path = "/content/face_landmarker.task"

urllib.request.urlretrieve(model_url, model_path)

print("Face Landmarker 모델 다운로드 완료!")

In [ ]:
import mediapipe as mp

BaseOptions = mp.tasks.BaseOptions
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

options = FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=model_path),
    running_mode=VisionRunningMode.IMAGE,
    num_faces=1,
    output_face_blendshapes=True
)

face_landmarker = FaceLandmarker.create_from_options(options)

print("MediaPipe Face Landmarker 준비 완료!")

In [ ]:
import cv2
import mediapipe as mp
import matplotlib.pyplot as plt

# 현재 분석할 이미지 사용
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# MediaPipe 이미지 객체 생성
mp_image = mp.Image(
    image_format=mp.ImageFormat.SRGB,
    data=image_rgb
)

# 얼굴 랜드마크 추출
mesh_results = face_landmarker.detect(mp_image)

if mesh_results.face_landmarks:

    face_landmarks = mesh_results.face_landmarks[0]

    print("얼굴 랜드마크 추출 성공!")
    print("추출된 랜드마크 수:", len(face_landmarks))

else:
    print("얼굴을 찾지 못했습니다.")

In [ ]:
# 입 주변 주요 랜드마크
mouth_points = [
    61, 291,       # 좌우 입꼬리
    13, 14,        # 입술 중앙 위아래
    78, 308,       # 입술 좌우
    82, 312,
    87, 317,
    95, 324
]

h, w = image_rgb.shape[:2]

landmark_image = image_rgb.copy()

for idx in mouth_points:

    landmark = face_landmarks[idx]

    x = int(landmark.x * w)
    y = int(landmark.y * h)

    cv2.circle(
        landmark_image,
        (x, y),
        4,
        (255, 0, 0),
        -1
    )

    cv2.putText(
        landmark_image,
        str(idx),
        (x + 5, y - 5),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.4,
        (0, 255, 0),
        1
    )

plt.figure(figsize=(8, 6))
plt.imshow(landmark_image)
plt.axis("off")
plt.title("MediaPipe Mouth Landmarks")
plt.show()

In [ ]:
# 좌우 입꼬리 좌표
left_corner = face_landmarks[61]
right_corner = face_landmarks[291]

# 입술 중앙 위아래 좌표
upper_lip = face_landmarks[13]
lower_lip = face_landmarks[14]

# 좌우 입꼬리 높이 차이
mouth_asymmetry = abs(
    left_corner.y - right_corner.y
)

# 입의 세로 벌어짐
mouth_opening = abs(
    upper_lip.y - lower_lip.y
)

# 입의 가로 길이
mouth_width = abs(
    left_corner.x - right_corner.x
)

# 얼굴 크기 기준 정규화
face_width = abs(
    face_landmarks[454].x - face_landmarks[234].x
)

asymmetry_ratio = mouth_asymmetry / face_width
opening_ratio = mouth_opening / face_width
width_ratio = mouth_width / face_width

print("===== 입 주변 특징 분석 =====")
print(f"좌우 비대칭 정도: {asymmetry_ratio:.4f}")
print(f"입 벌림 정도: {opening_ratio:.4f}")
print(f"입 가로 길이 비율: {width_ratio:.4f}")

In [ ]:
def extract_mouth_features(face_landmarks):

    left = face_landmarks[61]
    right = face_landmarks[291]
    upper = face_landmarks[13]
    lower = face_landmarks[14]

    face_width = abs(
        face_landmarks[454].x -
        face_landmarks[234].x
    )

    asymmetry = abs(left.y - right.y) / face_width
    mouth_opening = abs(upper.y - lower.y) / face_width
    mouth_width = abs(left.x - right.x) / face_width

    return {
        "mouth_asymmetry": asymmetry,
        "mouth_opening": mouth_opening,
        "mouth_width": mouth_width
    }

features = extract_mouth_features(face_landmarks)

print("===== 행복 분석용 얼굴 특징 =====")
for name, value in features.items():
    print(f"{name}: {value:.4f}")

In [ ]:
# 분노 분석에 사용할 랜드마크
anger_points = {
    "left_eyebrow": [70, 63, 105, 66, 107],
    "right_eyebrow": [336, 296, 334, 293, 300],
    "left_eye": [33, 133, 159, 145],
    "right_eye": [362, 263, 386, 374],
    "mouth": [61, 291, 13, 14]
}

# 랜드마크 좌표 추출
anger_features = {}

for name, indices in anger_points.items():
    coords = []

    for idx in indices:
        landmark = face_landmarks[idx]

        coords.append({
            "x": landmark.x,
            "y": landmark.y
        })

    anger_features[name] = coords

print("===== 분노 분석용 특징점 추출 =====")

for name, coords in anger_features.items():
    print(name, ":", len(coords), "개 추출")

In [ ]:
# 얼굴 너비 기준
face_width = abs(
    face_landmarks[454].x -
    face_landmarks[234].x
)

# 눈썹과 눈 사이 평균 거리
left_brow_eye = abs(
    face_landmarks[105].y -
    face_landmarks[159].y
)

right_brow_eye = abs(
    face_landmarks[334].y -
    face_landmarks[386].y
)

# 좌우 눈썹과 눈 사이 거리 정규화
brow_eye_distance = (
    (left_brow_eye + right_brow_eye) / 2
) / face_width

# 좌우 눈썹 높이 차이
brow_asymmetry = abs(
    face_landmarks[105].y -
    face_landmarks[334].y
) / face_width

print("===== 분노 분석 특징값 =====")
print(f"눈썹-눈 평균 거리: {brow_eye_distance:.4f}")
print(f"좌우 눈썹 비대칭: {brow_asymmetry:.4f}")

In [ ]:
import os
from glob import glob

happy_path = os.path.join(PROJECT_PATH, "processed", "Happy")

happy_images = []
for ext in ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"]:
    happy_images.extend(glob(os.path.join(happy_path, ext)))

happy_images = sorted(set(happy_images))

print("Happy 이미지 개수:", len(happy_images))
print("\n이미지 파일 예시:")
for path in happy_images[:10]:
    print(os.path.basename(path))

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import math

sample_images = happy_images[:20]

plt.figure(figsize=(16, 12))

for i, path in enumerate(sample_images):
    img = Image.open(path).convert("RGB")

    plt.subplot(4, 5, i + 1)
    plt.imshow(img)
    plt.title(os.path.basename(path)[:15], fontsize=8)
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
sample_images = happy_images[20:40]

plt.figure(figsize=(16, 12))

for i, path in enumerate(sample_images):
    img = Image.open(path).convert("RGB")

    plt.subplot(4, 5, i + 1)
    plt.imshow(img)
    plt.title(os.path.basename(path)[:15], fontsize=8)
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import cv2
import mediapipe as mp
import pandas as pd
import numpy as np
import os

BaseOptions = mp.tasks.BaseOptions
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

options = FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=model_path),
    running_mode=VisionRunningMode.IMAGE,
    num_faces=1
)

landmarker = FaceLandmarker.create_from_options(options)

results_list = []
failed = []

for path in happy_images:

    image = cv2.imread(path)

    if image is None:
        failed.append(path)
        continue

    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=image_rgb
    )

    result = landmarker.detect(mp_image)

    if not result.face_landmarks:
        failed.append(path)
        continue

    landmarks = result.face_landmarks[0]

    # 얼굴 너비로 정규화
    face_width = abs(landmarks[454].x - landmarks[234].x)

    if face_width == 0:
        failed.append(path)
        continue

    # 입꼬리 좌우 높이 차이
    mouth_asymmetry = abs(
        landmarks[61].y - landmarks[291].y
    ) / face_width

    # 입 벌림 정도
    mouth_opening = abs(
        landmarks[13].y - landmarks[14].y
    ) / face_width

    # 입 너비
    mouth_width = abs(
        landmarks[61].x - landmarks[291].x
    ) / face_width

    results_list.append({
        "filename": os.path.basename(path),
        "mouth_asymmetry": mouth_asymmetry,
        "mouth_opening": mouth_opening,
        "mouth_width": mouth_width
    })

landmarker.close()

happy_features = pd.DataFrame(results_list)

print("특징 추출 성공:", len(happy_features))
print("추출 실패:", len(failed))

display(happy_features.head())

# CSV 저장
save_path = os.path.join(PROJECT_PATH, "happy_features.csv")
happy_features.to_csv(save_path, index=False)

print("저장 위치:", save_path)

In [ ]:
import matplotlib.pyplot as plt

features = [
    "mouth_asymmetry",
    "mouth_opening",
    "mouth_width"
]

titles = [
    "Mouth Asymmetry",
    "Mouth Opening",
    "Mouth Width"
]

plt.figure(figsize=(15, 4))

for i, (feature, title) in enumerate(zip(features, titles)):
    plt.subplot(1, 3, i + 1)

    plt.hist(
        happy_features[feature],
        bins=20,
        edgecolor="black"
    )

    plt.title(title)
    plt.xlabel("Feature Value")
    plt.ylabel("Number of Images")

plt.tight_layout()
plt.show()

display(happy_features[features].describe().round(4))

In [ ]:
import cv2
import mediapipe as mp
import pandas as pd
import os

BaseOptions = mp.tasks.BaseOptions
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

options = FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=model_path),
    running_mode=VisionRunningMode.IMAGE,
    num_faces=1,
    output_face_blendshapes=True
)

landmarker = FaceLandmarker.create_from_options(options)

blendshape_results = []
failed_blendshape = []

for path in happy_images:

    image = cv2.imread(path)

    if image is None:
        failed_blendshape.append(path)
        continue

    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=image_rgb
    )

    result = landmarker.detect(mp_image)

    if not result.face_blendshapes:
        failed_blendshape.append(path)
        continue

    categories = result.face_blendshapes[0]

    row = {
        "filename": os.path.basename(path)
    }

    for category in categories:
        row[category.category_name] = category.score

    blendshape_results.append(row)

landmarker.close()

happy_blendshapes = pd.DataFrame(blendshape_results)

print("추출 성공:", len(happy_blendshapes))
print("추출 실패:", len(failed_blendshape))

display(happy_blendshapes.head())

save_path = os.path.join(
    PROJECT_PATH,
    "happy_blendshapes.csv"
)

happy_blendshapes.to_csv(save_path, index=False)

print("저장 위치:", save_path)

In [ ]:
# 주요 특징값 확인
features = [
    "mouthSmileLeft",
    "mouthSmileRight",
    "jawOpen",
    "mouthDimpleLeft",
    "mouthDimpleRight",
    "mouthStretchLeft",
    "mouthStretchRight"
]

display(
    happy_blendshapes[features]
    .describe()
    .round(4)
)

# 좌우 미소 차이 계산
happy_blendshapes["smile_asymmetry"] = abs(
    happy_blendshapes["mouthSmileLeft"]
    - happy_blendshapes["mouthSmileRight"]
)

# 결과 확인
print("좌우 미소 차이 통계")
display(
    happy_blendshapes["smile_asymmetry"]
    .describe()
    .round(4)
)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import os

# 비교할 특징과 사진 5장씩 선정
targets = {
    "좌우 미소 차이": (
        happy_blendshapes,
        "smile_asymmetry"
    ),
    "턱 벌림 정도": (
        happy_blendshapes,
        "jawOpen"
    ),
    "입꼬리 비대칭": (
        happy_features,
        "mouth_asymmetry"
    )
}

fig, axes = plt.subplots(3, 5, figsize=(17, 11))

for row, (title, (df, column)) in enumerate(targets.items()):

    top5 = df.nlargest(5, column)

    for col, (_, item) in enumerate(top5.iterrows()):

        filename = item["filename"]
        image_path = os.path.join(
            PROJECT_PATH,
            "processed",
            "Happy",
            filename
        )

        img = Image.open(image_path).convert("RGB")

        axes[row, col].imshow(img)
        axes[row, col].set_title(
            f"{column}\n{item[column]:.4f}",
            fontsize=9
        )
        axes[row, col].axis("off")

    axes[row, 0].set_ylabel(title, fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import os

# 두 특징 데이터 합치기
happy_merged = pd.merge(
    happy_features,
    happy_blendshapes,
    on="filename"
)

# 각 특징에서 상위 5장 선정
selected = pd.concat([
    happy_merged.nlargest(5, "smile_asymmetry"),
    happy_merged.nlargest(5, "jawOpen"),
    happy_merged.nlargest(5, "mouth_asymmetry")
])

# 중복 사진 제거
selected = selected.drop_duplicates(
    subset="filename"
).reset_index(drop=True)

print("대표 사진 수:", len(selected))

# 사람이 직접 붙일 라벨 열
selected["human_label"] = ""
selected["notes"] = ""

# 사진 출력
fig, axes = plt.subplots(3, 5, figsize=(17, 11))
axes = axes.flatten()

for i, (_, row) in enumerate(selected.iterrows()):

    path = os.path.join(
        PROJECT_PATH,
        "processed",
        "Happy",
        row["filename"]
    )

    img = Image.open(path).convert("RGB")

    axes[i].imshow(img)
    axes[i].set_title(
        f"{i+1}. {row['filename'][:15]}",
        fontsize=8
    )
    axes[i].axis("off")

for i in range(len(selected), 15):
    axes[i].axis("off")

plt.tight_layout()
plt.show()

# 라벨링용 CSV 저장
label_path = os.path.join(
    PROJECT_PATH,
    "happy_manual_labels.csv"
)

selected.to_csv(label_path, index=False)

print("라벨링 파일 저장:", label_path)

In [ ]:
# 16장 대표 사진 확인
print(selected[["filename"]].to_string(index=True))

# 라벨 목록
labels = [
    "일반적인 미소",
    "크게 웃음",
    "비대칭 미소",
    "비웃음 후보",
    "판단 어려움"
]

print("\n사용할 라벨:", labels)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import os

fig, axes = plt.subplots(4, 4, figsize=(16, 13))
axes = axes.flatten()

for i, (_, row) in enumerate(selected.iterrows()):

    path = os.path.join(
        PROJECT_PATH,
        "processed",
        "Happy",
        row["filename"]
    )

    img = Image.open(path).convert("RGB")

    axes[i].imshow(img)
    axes[i].set_title(
        f"{i+1}. {row['filename'][:18]}",
        fontsize=9
    )
    axes[i].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import os

# 두 데이터 합치기
happy_merged = pd.merge(
    happy_features,
    happy_blendshapes,
    on="filename"
)

# 특징값이 높은 사진 선정
top_asymmetry = happy_merged.nlargest(
    5, "smile_asymmetry"
)

top_jaw = happy_merged.nlargest(
    5, "jawOpen"
)

top_mouth = happy_merged.nlargest(
    5, "mouth_asymmetry"
)

# 특징값 기준 사진 15장 통합
top_samples = pd.concat([
    top_asymmetry,
    top_jaw,
    top_mouth
]).drop_duplicates(subset="filename")

# 나머지 사진 중 무작위 15장 선정
remaining = happy_merged[
    ~happy_merged["filename"].isin(top_samples["filename"])
]

random_samples = remaining.sample(
    n=15,
    random_state=42
)

# 최종 30장
validation_samples = pd.concat([
    top_samples,
    random_samples
]).drop_duplicates(subset="filename").reset_index(drop=True)

# 사람의 직접 판단을 기록할 열
validation_samples["human_label"] = ""
validation_samples["notes"] = ""

# 저장
validation_path = os.path.join(
    PROJECT_PATH,
    "happy_validation_30.csv"
)

validation_samples.to_csv(
    validation_path,
    index=False
)

print("검증용 사진:", len(validation_samples))
print("저장 위치:", validation_path)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import os

fig, axes = plt.subplots(6, 5, figsize=(17, 20))
axes = axes.flatten()

for i, (_, row) in enumerate(validation_samples.iterrows()):

    path = os.path.join(
        PROJECT_PATH,
        "processed",
        "Happy",
        row["filename"]
    )

    img = Image.open(path).convert("RGB")

    axes[i].imshow(img)
    axes[i].set_title(f"{i+1}. {row['filename'][:18]}", fontsize=8)
    axes[i].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# 사진 번호 순서대로 임시 라벨 지정
labels = [
    "판단 어려움", "일반적인 미소", "판단 어려움",
    "일반적인 미소", "크게 웃음",
    "크게 웃음", "일반적인 미소", "크게 웃음",
    "크게 웃음", "크게 웃음",
    "일반적인 미소", "크게 웃음", "일반적인 미소",
    "일반적인 미소", "크게 웃음",
    "일반적인 미소", "크게 웃음", "판단 어려움",
    "일반적인 미소", "일반적인 미소",
    "일반적인 미소", "크게 웃음", "판단 어려움",
    "판단 어려움", "일반적인 미소",
    "일반적인 미소", "일반적인 미소", "판단 어려움",
    "일반적인 미소", "일반적인 미소"
]

validation_samples["human_label"] = labels

validation_samples["notes"] = "화면 기반 임시 라벨"

validation_samples.to_csv(
    validation_path,
    index=False,
    encoding="utf-8-sig"
)

print("라벨 저장 완료!")
print(validation_samples["human_label"].value_counts())

display(
    validation_samples[
        ["filename", "human_label"]
    ]
)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 라벨과 특징값 결합
compare_df = validation_samples.copy()

# 그룹별 평균 특징값
features = [
    "jawOpen",
    "mouthSmileLeft",
    "mouthSmileRight",
    "smile_asymmetry",
    "mouth_asymmetry"
]

display(
    compare_df.groupby("human_label")[features]
    .mean()
    .round(4)
)

# 특징값별 그룹 비교 그래프
for feature in features:
    plt.figure(figsize=(7, 4))

    sns.boxplot(
        data=compare_df,
        x="human_label",
        y=feature
    )

    plt.title(feature)
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.show()

In [ ]:
print("실제 엑셀 열 이름:")
for col in df.columns:
    print(repr(col))

In [ ]:
# 분노 강도값 정렬
anger_values = anger_df["Intensity (0-1)"].dropna().sort_values()

print("분노 강도별 데이터")
display(anger_values.reset_index(drop=True))

# 강도 통계
print("\n강도 통계")
display(anger_values.describe().round(4))

# 분포 그래프
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.hist(anger_values, bins=10, edgecolor="black")
plt.title("Anger Intensity Distribution")
plt.xlabel("Intensity (0-1)")
plt.ylabel("Number of Images")
plt.show()

In [ ]:
print(df.columns.tolist())

In [ ]:
# 열 이름 앞뒤 공백 제거
df.columns = df.columns.str.strip()

# 분노 데이터 다시 추출
anger_df = df[
    df["Display"].astype(str).str.lower() == "anger"
].copy()

print("분노 데이터:", len(anger_df))

display(
    anger_df[
        ["Displayer ID", "Picture ID", "Intensity (0-1)", "FACS"]
    ].sort_values("Intensity (0-1)")
)

In [ ]:
import os

wsefep_path = os.path.join(
    PROJECT_PATH,
    "original",
    "WSEFEP"
)

anger_df["image_path"] = anger_df["Picture ID"].apply(
    lambda x: os.path.join(wsefep_path, x)
)

anger_df["file_exists"] = anger_df["image_path"].apply(
    os.path.exists
)

print("사진 연결 성공:", anger_df["file_exists"].sum())
print("사진 연결 실패:", (~anger_df["file_exists"]).sum())

display(
    anger_df[
        ["Picture ID", "Intensity (0-1)", "image_path", "file_exists"]
    ]
)

In [ ]:
import cv2
import mediapipe as mp
import pandas as pd
import numpy as np
import os

# MediaPipe Face Landmarker 설정
options = mp.tasks.vision.FaceLandmarkerOptions(
    base_options=mp.tasks.BaseOptions(
        model_asset_path=model_path
    ),
    running_mode=mp.tasks.vision.RunningMode.IMAGE,
    num_faces=1,
    output_face_blendshapes=True
)

landmarker = mp.tasks.vision.FaceLandmarker.create_from_options(
    options
)

anger_results = []
failed_images = []

# 사진이 존재하는 29장만 분석
valid_anger = anger_df[anger_df["file_exists"] == True]

for _, row in valid_anger.iterrows():

    image_path = row["image_path"]
    image = cv2.imread(image_path)

    if image is None:
        failed_images.append(row["Picture ID"])
        continue

    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=image_rgb
    )

    result = landmarker.detect(mp_image)

    if not result.face_landmarks:
        failed_images.append(row["Picture ID"])
        continue

    landmarks = result.face_landmarks[0]

    face_width = abs(
        landmarks[454].x - landmarks[234].x
    )

    if face_width == 0:
        failed_images.append(row["Picture ID"])
        continue

    # 기하학적 얼굴 특징
    left_brow_eye = abs(
        landmarks[105].y - landmarks[159].y
    ) / face_width

    right_brow_eye = abs(
        landmarks[334].y - landmarks[386].y
    ) / face_width

    brow_eye_distance = (
        left_brow_eye + right_brow_eye
    ) / 2

    brow_asymmetry = abs(
        landmarks[105].y - landmarks[334].y
    ) / face_width

    mouth_opening = abs(
        landmarks[13].y - landmarks[14].y
    ) / face_width

    mouth_asymmetry = abs(
        landmarks[61].y - landmarks[291].y
    ) / face_width

    features = {
        "Picture ID": row["Picture ID"],
        "Intensity": row["Intensity (0-1)"],
        "brow_eye_distance": brow_eye_distance,
        "brow_asymmetry": brow_asymmetry,
        "mouth_opening": mouth_opening,
        "mouth_asymmetry": mouth_asymmetry
    }

    # Blendshape 특징 추가
    for category in result.face_blendshapes[0]:
        features[category.category_name] = category.score

    anger_results.append(features)

landmarker.close()

# 데이터프레임 생성
anger_features = pd.DataFrame(anger_results)

print("특징 추출 성공:", len(anger_features))
print("추출 실패:", len(failed_images))
print("실패 파일:", failed_images)

display(anger_features.head())

# CSV 저장
anger_save_path = os.path.join(
    PROJECT_PATH,
    "anger_features.csv"
)

anger_features.to_csv(
    anger_save_path,
    index=False,
    encoding="utf-8-sig"
)

print("저장 위치:", anger_save_path)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr

# 분석할 주요 특징
features = [
    "brow_eye_distance",
    "brow_asymmetry",
    "mouth_opening",
    "mouth_asymmetry",
    "browDownLeft",
    "browDownRight",
    "eyeSquintLeft",
    "eyeSquintRight",
    "mouthPressLeft",
    "mouthPressRight",
    "jawOpen",
    "noseSneerLeft",
    "noseSneerRight"
]

# 실제 데이터에 존재하는 열만 선택
features = [
    col for col in features
    if col in anger_features.columns
]

# Spearman 상관계수와 p-value 계산
correlation_results = []

for feature in features:

    rho, p_value = spearmanr(
        anger_features["Intensity"],
        anger_features[feature]
    )

    correlation_results.append({
        "feature": feature,
        "correlation": rho,
        "p_value": p_value
    })

correlation_df = pd.DataFrame(correlation_results)

correlation_df["abs_correlation"] = (
    correlation_df["correlation"].abs()
)

correlation_df = correlation_df.sort_values(
    "abs_correlation",
    ascending=False
)

display(
    correlation_df.round(4)
)

# 상관관계 그래프
plt.figure(figsize=(10, 6))

sns.barplot(
    data=correlation_df,
    x="correlation",
    y="feature"
)

plt.axvline(0, color="black", linewidth=0.8)
plt.title("Anger Intensity - Facial Feature Correlation")
plt.xlabel("Spearman Correlation")
plt.ylabel("Facial Features")
plt.tight_layout()
plt.show()

In [ ]:
import re
from collections import Counter

# 분노 데이터의 FACS 정보 확인
facs_counts = Counter()

for facs in anger_df["FACS"].dropna():

    # AU 번호 추출
    au_numbers = re.findall(r'\d+', str(facs))

    for au in set(au_numbers):
        facs_counts["AU" + au] += 1

# 빈도 데이터프레임
facs_frequency = pd.DataFrame(
    facs_counts.items(),
    columns=["AU", "Frequency"]
)

facs_frequency["Percentage"] = (
    facs_frequency["Frequency"] / len(anger_df) * 100
)

facs_frequency = facs_frequency.sort_values(
    "Frequency",
    ascending=False
)

display(facs_frequency)

# 그래프
plt.figure(figsize=(10, 5))

sns.barplot(
    data=facs_frequency,
    x="Frequency",
    y="AU"
)

plt.title("FACS Action Unit Frequency in Anger Images")
plt.xlabel("Number of Images")
plt.ylabel("Action Unit")
plt.tight_layout()
plt.show()

In [ ]:
import re
import pandas as pd
from scipy.stats import spearmanr

# AU별 출현 여부 데이터 생성
au_data = []

for _, row in anger_df.iterrows():

    facs = str(row["FACS"])
    au_numbers = set(
        re.findall(r'(?<!\d)\d+(?!\d)', facs)
    )

    item = {
        "Picture ID": row["Picture ID"],
        "Intensity": row["Intensity (0-1)"]
    }

    for au in facs_counts.keys():
        au_num = au.replace("AU", "")
        item[au] = int(au_num in au_numbers)

    au_data.append(item)

au_df = pd.DataFrame(au_data)

# AU별 강도와의 상관관계
au_correlations = []

for au in facs_counts.keys():

    rho, p = spearmanr(
        au_df["Intensity"],
        au_df[au]
    )

    au_correlations.append({
        "AU": au,
        "Correlation": rho,
        "p_value": p
    })

au_corr_df = pd.DataFrame(au_correlations)

au_corr_df["abs_corr"] = (
    au_corr_df["Correlation"].abs()
)

au_corr_df = au_corr_df.sort_values(
    "abs_corr",
    ascending=False
)

display(
    au_corr_df.drop(columns="abs_corr").round(4)
)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# AU24 존재 여부
au_df["AU24_present"] = au_df["AU24"].map({
    0: "AU24 없음",
    1: "AU24 있음"
})

# 그룹별 강도 통계
display(
    au_df.groupby("AU24_present")["Intensity"]
    .agg(["count", "mean", "median", "std"])
    .round(4)
)

# 상자 그래프
plt.figure(figsize=(7, 4))

sns.boxplot(
    data=au_df,
    x="AU24_present",
    y="Intensity"
)

plt.title("Anger Intensity by AU24 Presence")
plt.xlabel("AU24")
plt.ylabel("Intensity (0-1)")
plt.show()

In [ ]:
from scipy.stats import mannwhitneyu

group_present = au_df[
    au_df["AU24"] == 1
]["Intensity"]

group_absent = au_df[
    au_df["AU24"] == 0
]["Intensity"]

# Mann-Whitney U 검정
statistic, p_value = mannwhitneyu(
    group_present,
    group_absent,
    alternative="two-sided"
)

print("AU24 있음 평균:", round(group_present.mean(), 4))
print("AU24 없음 평균:", round(group_absent.mean(), 4))
print("U 통계량:", statistic)
print("p-value:", round(p_value, 4))

In [ ]:
import re
from collections import Counter

au24_levels = []

for facs in anger_df["FACS"].dropna():

    matches = re.findall(
        r'24([A-E])',
        str(facs)
    )

    au24_levels.extend(matches)

print("AU24 강도 표시 빈도")

display(
    pd.Series(au24_levels)
    .value_counts()
    .sort_index()
)

In [ ]:
import re
import pandas as pd
from scipy.stats import spearmanr

# AU24 강도와 전체 분노 강도 연결
au24_data = []

for _, row in anger_df.iterrows():

    facs = str(row["FACS"])

    match = re.search(r'24([A-E])', facs)

    if match:
        au24_data.append({
            "Picture ID": row["Picture ID"],
            "Intensity": row["Intensity (0-1)"],
            "AU24_Level": match.group(1)
        })

au24_df = pd.DataFrame(au24_data)

# 강도 수준을 순서형 숫자로 변환
level_mapping = {
    "A": 1,
    "B": 2,
    "C": 3,
    "D": 4,
    "E": 5
}

au24_df["AU24_Level_Number"] = (
    au24_df["AU24_Level"].map(level_mapping)
)

# 단계별 전체 분노 강도 통계
summary = au24_df.groupby("AU24_Level").agg(
    count=("Intensity", "count"),
    mean_intensity=("Intensity", "mean"),
    median_intensity=("Intensity", "median")
).reindex(["A", "B", "C", "D", "E"])

display(summary.round(4))

# AU24 세부 강도와 전체 분노 강도의 상관관계
rho, p_value = spearmanr(
    au24_df["AU24_Level_Number"],
    au24_df["Intensity"]
)

print("Spearman 상관계수:", round(rho, 4))
print("p-value:", round(p_value, 4))

In [ ]:
import torch
import torch.nn as nn
from torchvision import models
import os

# 학습된 모델 불러오기
model_path = os.path.join(
    PROJECT_PATH,
    "model",
    "best_emotion_model.pth"
)

checkpoint = torch.load(
    model_path,
    map_location=device,
    weights_only=False
)

class_names = checkpoint["class_names"]

# EfficientNet 모델 구조
model = models.efficientnet_b0(weights=None)

num_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_features, 7)

model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(device)
model.eval()

print("모델 로드 성공!")
print("학습 당시 최고 Epoch:", checkpoint["epoch"])
print("검증 정확도:", checkpoint["val_acc"])
print("감정 종류:", class_names)

In [ ]:
from google.colab import files
import os

uploaded = files.upload()

test_images = list(uploaded.keys())

print("업로드된 사진:", len(test_images))

for path in test_images:
    print(path)

In [ ]:
import os
import pandas as pd

# 기존 분할 데이터 불러오기
train_df = pd.read_csv(os.path.join(split_path, "train.csv"))
val_df = pd.read_csv(os.path.join(split_path, "validation.csv"))
test_df = pd.read_csv(os.path.join(split_path, "test.csv"))

print("기존 데이터 분할 확인")
print("----------------------")
print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))
print("Total:", len(train_df) + len(val_df) + len(test_df))

print("\n감정별 학습 데이터")
print(train_df["emotion"].value_counts().sort_index())

print("\n감정별 검증 데이터")
print(val_df["emotion"].value_counts().sort_index())

print("\n감정별 테스트 데이터")
print(test_df["emotion"].value_counts().sort_index())

In [ ]:
import os
import cv2
import pandas as pd
from tqdm.auto import tqdm

# 얼굴 이미지 저장 경로
FACE_PATH = os.path.join(PROJECT_PATH, "face_processed")
os.makedirs(FACE_PATH, exist_ok=True)

# 감정별 폴더 생성
for emotion in class_names:
    os.makedirs(
        os.path.join(FACE_PATH, emotion),
        exist_ok=True
    )

# 기존 데이터 분할 불러오기
split_dfs = {
    "train": train_df,
    "validation": val_df,
    "test": test_df
}

face_records = []
fail_records = []

# YOLO 얼굴 추출
for split_name, df in split_dfs.items():

    print(f"\n[{split_name}] 얼굴 추출 시작")

    for _, row in tqdm(df.iterrows(), total=len(df)):

        image_path = row["path"]
        emotion = row["emotion"]

        image = cv2.imread(image_path)

        if image is None:
            fail_records.append(image_path)
            continue

        results = face_detector(image)
        boxes = results[0].boxes

        if len(boxes) == 0:
            fail_records.append(image_path)
            continue

        # 가장 높은 신뢰도의 얼굴 선택
        best_idx = boxes.conf.argmax().item()
        box = boxes[best_idx]

        x1, y1, x2, y2 = map(
            int, box.xyxy[0].tolist()
        )

        h, w = image.shape[:2]

        # 얼굴 주변 30% 여백
        face_w = x2 - x1
        face_h = y2 - y1

        margin_x = int(face_w * 0.3)
        margin_y = int(face_h * 0.3)

        x1 = max(0, x1 - margin_x)
        y1 = max(0, y1 - margin_y)
        x2 = min(w, x2 + margin_x)
        y2 = min(h, y2 + margin_y)

        face_crop = image[y1:y2, x1:x2]

        if face_crop.size == 0:
            fail_records.append(image_path)
            continue

        # 분할별 폴더 생성
        save_dir = os.path.join(
            FACE_PATH, split_name, emotion
        )
        os.makedirs(save_dir, exist_ok=True)

        filename = os.path.basename(image_path)
        save_path = os.path.join(save_dir, filename)

        cv2.imwrite(save_path, face_crop)

        face_records.append({
            "path": save_path,
            "emotion": emotion,
            "split": split_name,
            "original_path": image_path
        })

# 결과 CSV 저장
face_df = pd.DataFrame(face_records)

face_df.to_csv(
    os.path.join(FACE_PATH, "face_dataset.csv"),
    index=False
)

print("\n얼굴 추출 완료")
print("성공:", len(face_records))
print("실패:", len(fail_records))
print("전체:", len(train_df) + len(val_df) + len(test_df))

In [ ]:
from google.colab import drive # 노트북 껐다켜서 다시 실행

drive.mount('/content/drive')

In [ ]:
import os

# 프로젝트 폴더 경로
PROJECT_PATH = "/content/drive/MyDrive/Face_Emotion_Project"

# 데이터 및 모델 폴더
split_path = os.path.join(PROJECT_PATH, "split")
output_path = os.path.join(PROJECT_PATH, "processed")
FACE_PATH = os.path.join(PROJECT_PATH, "face_processed")
model_path = os.path.join(PROJECT_PATH, "model")

print("프로젝트 폴더 확인:", os.path.exists(PROJECT_PATH))
print("얼굴 데이터 확인:", os.path.exists(FACE_PATH))
print("모델 폴더 확인:", os.path.exists(model_path))

In [ ]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("PyTorch 버전:", torch.__version__)
print("GPU 사용 가능:", torch.cuda.is_available())
print("사용 장치:", device)

if torch.cuda.is_available():
    print("GPU 이름:", torch.cuda.get_device_name(0))

In [ ]:
!pip install -q ultralytics

In [ ]:
import os
import cv2
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image
from tqdm.auto import tqdm
from ultralytics import YOLO

from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader

# 감정 클래스
class_names = [
    "Angry", "Disgust", "Fear",
    "Happy", "Neutral", "Sad", "Surprise"
]

# YOLO 얼굴 감지 모델
face_detector = YOLO(
    "https://github.com/akanametov/yolo-face/releases/download/1.0.0/yolov11n-face.pt"
)

print("YOLO 모델 불러오기 완료!")

In [ ]:
face_csv_path = os.path.join(
    FACE_PATH, "face_dataset.csv"
)

print("얼굴 데이터 CSV 존재:", os.path.exists(face_csv_path))

if os.path.exists(face_csv_path):
    face_df = pd.read_csv(face_csv_path)

    print("얼굴 데이터 개수:", len(face_df))
    print("\n분할별 데이터:")
    print(face_df["split"].value_counts())

    print("\n감정별 데이터:")
    print(face_df["emotion"].value_counts().sort_index())

In [ ]:
# 감정 라벨 숫자 변환
label_to_idx = {
    emotion: idx
    for idx, emotion in enumerate(class_names)
}

# 학습용 전처리
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# 검증 및 테스트용 전처리
eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# 데이터셋 클래스
class FaceEmotionDataset(Dataset):

    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]

        image = Image.open(row["path"]).convert("RGB")
        label = label_to_idx[row["emotion"]]

        if self.transform:
            image = self.transform(image)

        return image, label

# 데이터 분할
face_train_df = face_df[face_df["split"] == "train"].reset_index(drop=True)
face_val_df = face_df[face_df["split"] == "validation"].reset_index(drop=True)
face_test_df = face_df[face_df["split"] == "test"].reset_index(drop=True)

# 데이터셋 생성
face_train_dataset = FaceEmotionDataset(face_train_df, train_transform)
face_val_dataset = FaceEmotionDataset(face_val_df, eval_transform)
face_test_dataset = FaceEmotionDataset(face_test_df, eval_transform)

# DataLoader 생성
face_train_loader = DataLoader(
    face_train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

face_val_loader = DataLoader(
    face_val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

face_test_loader = DataLoader(
    face_test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# 확인
images, labels = next(iter(face_train_loader))

print("Train:", len(face_train_dataset))
print("Validation:", len(face_val_dataset))
print("Test:", len(face_test_dataset))
print("이미지 크기:", images.shape)
print("라벨 크기:", labels.shape)
print("첫 번째 라벨:", class_names[labels[0].item()])

In [ ]:
# EfficientNet-B0 생성
weights = models.EfficientNet_B0_Weights.DEFAULT
face_model = models.efficientnet_b0(weights=weights)

# 7가지 감정 분류층으로 변경
num_features = face_model.classifier[1].in_features

face_model.classifier[1] = nn.Linear(
    num_features, 7
)

face_model = face_model.to(device)

# 손실 함수 및 최적화 함수
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    face_model.parameters(),
    lr=0.0001,
    weight_decay=0.0001
)

# 학습 설정
num_epochs = 20
best_val_acc = 0.0

face_model_path = os.path.join(
    PROJECT_PATH,
    "model",
    "face_best_emotion_model.pth"
)

history = []

# 학습 시작
for epoch in range(num_epochs):

    face_model.train()

    train_loss = 0
    train_correct = 0
    train_total = 0

    for images, labels in tqdm(
        face_train_loader,
        desc=f"Epoch {epoch+1}/{num_epochs}"
    ):

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = face_model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)

        _, predicted = torch.max(outputs, 1)

        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()

    train_loss /= train_total
    train_acc = train_correct / train_total

    # 검증
    face_model.eval()

    val_loss = 0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for images, labels in face_val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = face_model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)

            _, predicted = torch.max(outputs, 1)

            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_loss /= val_total
    val_acc = val_correct / val_total

    print(f"\nEpoch [{epoch+1}/{num_epochs}]")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Train Accuracy: {train_acc:.4f}")
    print(f"Validation Loss: {val_loss:.4f}")
    print(f"Validation Accuracy: {val_acc:.4f}")

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc
    })

    # 최고 성능 모델 저장
    if val_acc > best_val_acc:

        best_val_acc = val_acc

        torch.save({
            "model_state_dict": face_model.state_dict(),
            "class_names": class_names,
            "epoch": epoch + 1,
            "val_acc": val_acc
        }, face_model_path)

        print("최고 성능 모델 저장 완료!")

print("\n얼굴 이미지 모델 학습 완료!")
print("최고 Validation Accuracy:", best_val_acc)
print("저장 경로:", face_model_path)